In [1]:
# QP1-1 Grading Tag:

import random
from abc import ABC, abstractmethod

# This allows animals like Peacock to move randomly
# To ensure that all animals follow the same required structure, we utilize the ABC and and abstractmethod to create abstract base classes

# Now, we create the LocalView class

# This class represents what the animal sees during its turn to move
# In other word, this class informs the animal: "these are the directions you are allowed to move right now."

# Also, the class prevents the animals from moving outside the zoo bounderies, and into another animal's space
# This class hierarchy is structured clearly for posterity; this could inform future developers and help them extend the zoo with new animals

class LocalView:
    def __init__(self, can_move_north, can_move_east, can_move_south, can_move_west):
        
        # This ensures that the animal always has the option to stay in place
        # Therefore, we start incorporating the term "stay" as a valid movement option
        self.move_directions = ['stay']
        
        # Now, we only add directions if movement is allowed
        if can_move_north:
            self.move_directions.append('north')
        if can_move_east:
            self.move_directions.append('east')
        if can_move_south:
            self.move_directions.append('south')
        if can_move_west:
            self.move_directions.append('west')
            
# Here we create the abstract animal class

# The abstract class is considered as the most important class in the entire project because:
# - we never create animal objects directly
# - other animals are supposed to inherit from this class
# - it sets a defined structure that requires all animals to follow

class Animal(ABC):
    
    def __init__(self):
        # each animal is visually represented by an icon in the zoo grid: for example - P for Peacock, C for Cow
        self._icon = '?'
    
    # Now, we create the icon getter function what will allow us to safely retrieve the animal's icon.
    @property
    def icon(self):
        return self._icon
    # Now, we create the icon setter to ensure that the icon is exactly one character preventing bugs while keeping the grid clean
    @icon.setter
    def icon(self, value):
        
        if len(value) != 1:
            raise Exception("icon must be a single character.")
        self._icon = value # creating a protected instance attribute
        
    # Here we create a String and Representation method functions
    def __str__(self):
        return self.icon
    
    def __repr__(self):
        class_name = type(self).__name__ # this will help us get the name of subclasses, e.g. Peacock, Cow, e.t.c.
        # Now we show the icon
        icon_info = "icon" + str(self.icon)
        # here we combine both the class name and the icon into a readable string
        output = class_name + "(" + icon_info + ")"
        
        return output
    
    # By creating the abstract move method, we enforce Polymorphism to ensure that this method is implemented by all sub-classes and that each animal moves in it's own unique way
    @abstractmethod
    def move(self, local_view):
        pass
    
# Now, we create the Peacock Class:

# We know that the Peacock inherits from the Parent class Animal and would automatically gain all Animal functionality
# The restriction here, is we need to customize its movement behavior

class Peacock(Animal):
    def __init__(self):
        
        # Here we call the parent constructor and assign the peacock's icon
        super().__init__()
        self.icon = '\U0001F99A'
    
    def move(self, local_view):
        
        # In this function, the Peacock moves randomly and would choose any available direction from its LocalView
        direction = random.choice(local_view.move_directions)
        return direction
    
# Here we create the Cow Class:

# The cow has a very simple movement rule compared to the Peacock and prefers moving north
# If north is available it stays in that location

class Cow(Animal):
    def __init__(self):
        
        # calling the parent constructor and assigning the cow's icon
        super().__init__()
        self.icon = '\U0001F404'
        
    def move(self, local_view):
        
        # First, we need to check if the north location is allowed
        if 'north' in local_view.move_directions:
            return 'north'
        return 'stay'
    
# Now, we create the zoo class:

# The zoo class manages the entire simulation because it is responsible for storing animals, tracking positions, updating movement and printing the grid

class Zoo:
    def __init__(self, height, width):
        # we create the instance parameters:
        # height = number of rows 
        # width = number of columns
        
        self.height = height
        self.width = width
        
        # This forms a protected dictionary that stores animal positions
        self._animals = {} 
        
    def add_animal(self, animal, row, col):
        
        # this enables us to place an animal at as specific location
        self._animals[animal] = (row, col)
        
    # here we are returning the (row, col) position of the given animal 
    def position_of_animal(self, animal):
        return self._animals.get(animal)
        
    # this function would help us identify a specific animal's position
    def animal_at_position(self, row, col):
        for animal, position in self._animals.items():
            
            if position == (row, col):
                return animal
        return None
    # time_step simulation 

    def time_step(self):
        # Now we create a copy of the animal list where each animal moves one at a time

        animals = list(self._animals.keys())
        for animal in animals:
            row, col = self._animals[animal]

            #Now, we determine which animals are allowed
            can_move_north = (row > 0 and self.animal_at_position(row - 1, col) is None)
            can_move_east = (col < self.width - 1 and self.animal_at_position(row, col + 1) is None)
            can_move_south = (row < self.height - 1 and self.animal_at_position(row + 1, col) is None)
            can_move_west = (col > 0 and self.animal_at_position(row, col - 1) is None)

            # creating the LocalView object 
            local_view = LocalView(can_move_north, can_move_east, can_move_south, can_move_west)

            direction = animal.move(local_view) # this decides where to move

            new_row, new_col = row, col

            if direction == 'north':
                new_row -= 1
            elif direction == 'east':
                new_col += 1
            elif direction == 'south':
                new_row += 1
            elif direction == 'west':
                new_col -= 1

            # After setting the animal directions, we update position
            self._animals[animal] = (new_row, new_col)
            
    # Now, we need to print the zoo grid

    def __str__(self):

        """Return a string representation of the zoo grid.
        Empty spots are shown as '.' animals show their icons.
        """

        grid = []

        for i in range(self.height):
            row = []

            for j in range(self.width):
                row.append('.') # we are wanting to fill the empty spots with dots
            grid.append(row)

        for animal, position in self._animals.items():

            row, col = position

            grid[row][col] = str(animal)

        # converting the grid to string

        output = "" # this will store the final string

        for row in grid:
            row_string = ""

            for cell in row:
                row_string += cell + " "
            row_string = row_string.rstrip() # remove trailing space at the end
            output += row_string + "\n"
        return output
    
    def __repr__(self):
        """Return a detailed representation of the Zoo for debugging purposes."""
        
        # here, we are wanting to start with the class name
        output = "Zoo(\n"
        output += " Height: " + str(self.height) + "\n"
        output += " Width: " + str(self.width) + "\n"
        
        # here we also list all animals in the zoo and their positions
        output += " Animals: \n"
        for animal, position in self._animals.items():
            
            output += "   " + str(animal)
            output += " at position" + str(position) + "\n"
        output += ")"
        
        return output
    
# Finally, we need to run the simulation

# creating the zoo
zoo = Zoo(height=6, width=6)
#creating animals
peacock = Peacock()
cow = Cow()

# adding animals
zoo.add_animal(peacock, 2, 2)
zoo.add_animal(cow, 4, 4)

for step in range(10):
    print("Step", step)
    print(zoo)
    
    zoo.time_step()

Step 0
. . . . . .
. . . . . .
. . 🦚 . . .
. . . . . .
. . . . 🐄 .
. . . . . .

Step 1
. . . . . .
. . . . . .
. 🦚 . . . .
. . . . 🐄 .
. . . . . .
. . . . . .

Step 2
. . . . . .
. . . . . .
🦚 . . . 🐄 .
. . . . . .
. . . . . .
. . . . . .

Step 3
. . . . . .
. . . . 🐄 .
. . . . . .
🦚 . . . . .
. . . . . .
. . . . . .

Step 4
. . . . 🐄 .
. . . . . .
. . . . . .
🦚 . . . . .
. . . . . .
. . . . . .

Step 5
. . . . 🐄 .
. . . . . .
. . . . . .
🦚 . . . . .
. . . . . .
. . . . . .

Step 6
. . . . 🐄 .
. . . . . .
🦚 . . . . .
. . . . . .
. . . . . .
. . . . . .

Step 7
. . . . 🐄 .
. . . . . .
🦚 . . . . .
. . . . . .
. . . . . .
. . . . . .

Step 8
. . . . 🐄 .
. . . . . .
🦚 . . . . .
. . . . . .
. . . . . .
. . . . . .

Step 9
. . . . 🐄 .
. . . . . .
. 🦚 . . . .
. . . . . .
. . . . . .
. . . . . .



In [ ]:
# Ungraded testing code for viewing the zoo

from IPython import display 
import time

LOOP_TIME_SECONDS = 10
REFRESH_TIME_SECONDS = .5

# here we are creating the zoo
zoo = Zoo(height=6, width=6)
# we have two subclasses - Peacock and the cow - so, we now create individual animal objects
peacock = Peacock()
cow = Cow()

# Now, we add the animals to specific positions in the zoo
zoo.add_animal(peacock, 2, 2)
zoo.add_animal(cow, 4, 4)

# running the simulation
num_steps = int(LOOP_TIME_SECONDS/REFRESH_TIME_SECONDS)

for i in range(int(LOOP_TIME_SECONDS/REFRESH_TIME_SECONDS)):
    display.clear_output(wait=True)
    
    print("Step", step)
    
    zoo.time_step()
    print(zoo)
    
    time.sleep(REFRESH_TIME_SECONDS)